# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a guide for loading and exploring the [FAIR²](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset source is provided via a Croissant schema URL. This dataset examines protein abundance, modifications, and sequences in human samples collected using mass spectrometry analysis.

In [ ]:
# Ensure `mlcroissant` and common analysis packages are installed
!pip install mlcroissant[full] pandas matplotlib seaborn

## 1. Data Loading
Load metadata and records from the FAIR² dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Define the dataset Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset Title: {metadata.name}")
print(f"Description: {metadata.description}\n\nPublished: {metadata.datePublished}\nVersion: {metadata.version}")
print(f"License: {metadata.license}")

## 2. Data Overview
Review available record sets (`cr:RecordSet`), fields, and their `@id` properties. This helps you select which record sets and fields to analyze. All references will use the entities' `@id` for clear and reproducible access.

_Note: In Croissant datasets, record sets contain the tabular data (like tables or data files) and each field describes a column. To explore all data elements, we will enumerate `RecordSet` entities and their fields by `@id`._

In [ ]:
# List available record sets and their fields, referencing them by @id

record_sets = list(metadata.record_sets)

print(f"Number of record sets: {len(record_sets)}")
for rec in record_sets:
    print(f"RecordSet @id: {rec['@id']}")
    print(f"  Name: {rec.get('name', '[no name]')}")
    print(f"  Description: {rec.get('description', '[no description]')}")
    print(f"  Fields (by @id):")
    for field in rec['fields']:
        print(f"    - {field['@id']} (name: {field.get('name', '[no name]')}, dataType: {field.get('dataType', '[n/a]')})")
    print()

## 3. Data Extraction
Load data from a specific record set (by `@id`) into a Pandas DataFrame for analysis. 
First, select a record set and review the field names (columns), then demonstrate how to extract its contents for analysis.

_Below, you can adjust `record_sets_ids` to focus on any subset of record sets._

In [ ]:
# Choose record sets to analyze (by @id)
record_sets_ids = [rec['@id'] for rec in record_sets]
dataframes = {}

for rec_id in record_sets_ids:
    print(f"Loading records for record set {rec_id} ...")
    records = list(dataset.records(record_set=rec_id))
    df = pd.DataFrame(records)
    dataframes[rec_id] = df
    print(f"Loaded {len(df)} records with columns:")
    print(df.columns.tolist(), "\n")
# For demonstration, select the first available record set
if record_sets_ids:
    main_record_set_id = record_sets_ids[0]
    main_df = dataframes[main_record_set_id]
    print(f"Preview of the first five rows of record set: {main_record_set_id}")
    display(main_df.head())

## 4. Exploratory Data Analysis (EDA)
Apply data processing steps such as filtering, normalization, and grouping over specific fields. 

_To ensure reproducibility, we'll reference columns by their Croissant field `@id`, as shown in previous steps. Make sure to adapt the field IDs below to match those in the record set you're analyzing (see column listing above)._

In [ ]:
# Identify a numeric field and a group field by @id
# You may want to adapt these field IDs based on the content printed above.
# Example (change as appropriate):
# numeric_field_id = "acc.mw"   # mass/weight column
# group_field_id = "acc.sample_id"  # group by sample

# Find possible numeric fields
numeric_candidates = [col for col in main_df.columns if main_df[col].dtype in [float, int]]
print(f"Numeric candidates: {numeric_candidates}")
if numeric_candidates:
    numeric_field_id = numeric_candidates[0]
else:
    numeric_field_id = None

# Try to find a grouping field
group_candidates = [col for col in main_df.columns if col != numeric_field_id]
group_field_id = group_candidates[0] if group_candidates else None

if numeric_field_id:
    print(f"Using numeric field: {numeric_field_id}")
    # Example threshold for demonstration
    threshold = main_df[numeric_field_id].mean() if main_df[numeric_field_id].dtype in [float, int] else 10
    filtered_df = main_df[main_df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    print(filtered_df.head())

    norm_col = f"{numeric_field_id}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, norm_col]].head())

    if group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
        print(f"Grouped data by {group_field_id}: (mean values)")
        print(grouped_df.head())
else:
    print("No numeric field found for analysis.")

## 5. Visualization
Visualize the distribution of the chosen numeric field and its normalized values, group-wise if relevant.

_Note: Visualizations are made with `matplotlib` and `seaborn`._

In [ ]:
if numeric_field_id and not main_df[numeric_field_id].isna().all():
    plt.figure(figsize=(7,4))
    sns.histplot(main_df[numeric_field_id].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field_id} in {main_record_set_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()
    
    if group_field_id and group_field_id in main_df.columns:
        plt.figure(figsize=(8, 5))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=main_df)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
In this notebook, we've loaded and explored the FAIR² mass spectrometry dataset using the `mlcroissant` library, referencing all variables by their Croissant `@id` for transparency and reproducibility. 

Key steps involved:
- Listing record sets and their field `@id`s
- Extracting tabular data and examining columns
- Selecting and processing numeric data fields
- Visualizing key distributions in the dataset

You can further extend this notebook by exploring additional record sets, analyzing other numeric or categorical fields (all by `@id`), or applying machine learning techniques to the processed data. For more information, see the [mlcroissant documentation](https://github.com/mlcommons/croissant) and the [FAIR² dataset description](https://sen.science/doi/10.71728/senscience.vq4a-28xa).
